# Trabajo N°2 — Fundamentos de Ciencia de Datos
**Proyecto 3 — Predicción del valor de propiedades (Melbourne 2016 → 2017)**

**Hito 2 — Presentación oral final: modelo, método, resultados y conclusión**

Autores: C. Abrigo · C. Herrera · K. Urbina

Este notebook **continúa** el Trabajo 1 (`V2_Trabajo_N1_FUNDAMENTOS`). No vuelve a hacer el EDA. Parte de la hipótesis ya formulada y la pone a prueba con validación temporal.

## Foco y contexto (qué responde el Hito 2)

El enunciado del curso pide una oral final que, **además** del análisis exploratorio, contenga: el(los) modelo(s), la metodología de ajuste, los resultados y la conclusión. No hay informe escrito.

| Ya cerrado en el Hito 1 (Trabajo 1) | Lo que hace este notebook |
|---|---|
| Pregunta: ¿2016 tiene señal para tasar 2017? | Entrenar **solo 2016** y evaluar en **2017** |
| H₁: esas relaciones superan una estimación básica | La estimación básica es la **mediana de Precio 2016** |
| H₀: no superan esa estimación | Rechazar o no rechazar H₀ con R², MAE y RMSE en 2017 |
| 172 barrios nuevos; +1,1% de mediana agregada | Medir **dónde se rompe** el modelo (barrios no vistos, cola de precio) |

**Frase puente:** en el Hito 1 vimos señal en habitaciones, tipo y ubicación. Aquí no redescubrimos eso: preguntamos si un modelo aprendido en 2016 **predice 2017 mejor que una mediana**, y con qué error de negocio.

2017 llega al **23 de septiembre**: no es un año calendario completo.

**Paso 1 — Importación de librerías**

**¿Qué?** Cargar pandas, scikit-learn y gráficos, con la misma semilla en todos los modelos.

**¿Por qué?** Reproducibilidad: cualquier cifra del oral debe poder salir de una corrida de este notebook.

In [ ]:
from pathlib import Path
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GridSearchCV, KFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeRegressor

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 140)

RANDOM_STATE = 42

**Paso 2 — Carga del mismo `housing_data.csv` del Trabajo 1**

**¿Qué?** Una sola carga controlada. No se inventan filas ni se reemplaza el archivo.

**¿Por qué?** El Hito 2 se evalúa sobre el mismo insumo del Hito 1 (13.580 × 21).

In [ ]:
DATA_CANDIDATES = [
    Path("housing_data.csv"),
    Path("/content/housing_data.csv"),
]

DATA_PATH = next((p for p in DATA_CANDIDATES if p.exists()), None)
DATA_AVAILABLE = DATA_PATH is not None

if DATA_AVAILABLE:
    df = pd.read_csv(DATA_PATH)
    print(f"Dataset cargado desde: {DATA_PATH}")
    print(f"Filas: {df.shape[0]:,} | Columnas: {df.shape[1]}")
    print("Columnas originales:", df.columns.tolist())
else:
    df = pd.DataFrame()
    print("NO VERIFICABLE CON LOS DATOS DISPONIBLES: no se encontró housing_data.csv.")

**Paso 3 — Homogeneización heredada del Hito 1**

**¿Qué?** Los mismos nombres en español, las mismas recodificaciones de `Tipo` y `Método`, y `Fecha` con `dayfirst=True`.

**¿Por qué?** El oral no puede cambiar de léxico entre notebooks. `casa` / `casapareada` / `duplex` ya se usaron en el EDA.

In [ ]:
if not DATA_AVAILABLE:
    print("NO VERIFICABLE CON LOS DATOS DISPONIBLES.")
else:
    diccionario_columnas = {
        "Suburb": "Barrio",
        "Address": "Dirección",
        "Rooms": "Habitaciones",
        "Type": "Tipo",
        "Price": "Precio",
        "Method": "Método",
        "SellerG": "Vendedor",
        "Date": "Fecha",
        "Distance": "Distancia",
        "Postcode": "Postcode",
        "Bedroom2": "Habitaciones2",
        "Bathroom": "Baños",
        "Car": "Automóviles",
        "Landsize": "Tamaño_Tierra",
        "BuildingArea": "Tamaño_Construcción",
        "YearBuilt": "Año_Construcción",
        "CouncilArea": "Comuna",
        "Lattitude": "Latitud",
        "Longtitude": "Longitud",
        "Regionname": "Región",
        "Propertycount": "Cantidad_Propiedades",
    }
    df = df.rename(columns=diccionario_columnas)

    df["Tipo"] = df["Tipo"].replace({"h": "casa", "t": "casapareada", "u": "duplex"})
    df["Método"] = df["Método"].replace({
        "S": "Vendida",
        "SP": "Vendida_anteriormente",
        "PI": "Pasada_en_remate",
        "VB": "Oferta_vendedor",
        "SA": "Rematada",
    })

    df["Fecha"] = pd.to_datetime(df["Fecha"], dayfirst=True, errors="raise")
    df["Year"] = df["Fecha"].dt.year

    print("Rango de fechas:", df["Fecha"].min().date(), "→", df["Fecha"].max().date())
    print(df["Year"].value_counts().sort_index())
    print("\nInterpretación: 2017 queda truncado al 23-sep; no se describe como año calendario completo.")

**Paso 4 — Decisiones de variables (heredadas del EDA, no reabiertas)**

**¿Qué?** Recordar qué entra al modelo y qué no. Una auditoría corta, no el EDA otra vez.

**¿Por qué?** El Hito 1 ya justificó los descartes. Aquí solo se comprueba que el archivo no cambió y se fija la matriz $X$.

In [ ]:
if not DATA_AVAILABLE:
    print("NO VERIFICABLE CON LOS DATOS DISPONIBLES.")
else:
    nulos = (
        df[["Tamaño_Construcción", "Año_Construcción", "Comuna", "Automóviles",
            "Latitud", "Longitud", "Distancia", "Postcode"]]
        .isna().sum()
        .rename("nulos")
    )
    print("Nulos (comprobación, no un EDA nuevo):")
    print(nulos.to_string())

    barrios_2016 = set(df.loc[df["Year"] == 2016, "Barrio"])
    barrios_2017 = set(df.loc[df["Year"] == 2017, "Barrio"])
    barrios_nuevos_2017 = barrios_2017 - barrios_2016
    n_filas_nuevas = int(
        ((df["Year"] == 2017) & df["Barrio"].isin(barrios_nuevos_2017)).sum()
    )
    print(f"\nBarrios nuevos en 2017: {len(barrios_nuevos_2017)} ({n_filas_nuevas} filas)")
    print(
        "Ubicación en el modelo: Distancia + Latitud + Longitud + Región. "
        "No Barrio one-hot (niveles nuevos). No Postcode (mismo problema, más granular)."
    )
    print(
        "Nulos espaciales: Latitud/Longitud/Distancia = 0. "
        "Comuna nula es fenómeno de 2017; no se imputa en este notebook."
    )

**Paso 5 — Partición temporal estricta**

**¿Qué?** Train = 2016. Test = 2017. La variable objetivo es `Precio` en AUD.

**¿Por qué?** El Proyecto 3 exige predicción fuera de tiempo. Un split aleatorio mezclaría el futuro con el entrenamiento (data leakage).

In [ ]:
if not DATA_AVAILABLE:
    print("NO VERIFICABLE CON LOS DATOS DISPONIBLES.")
else:
    variables_numericas = [
        "Habitaciones",
        "Baños",
        "Automóviles",
        "Distancia",
        "Tamaño_Tierra",
        "Latitud",
        "Longitud",
        "Cantidad_Propiedades",
    ]
    variables_categoricas = ["Tipo", "Método", "Región"]
    features = variables_numericas + variables_categoricas

    train_mask = df["Year"] == 2016
    test_mask = df["Year"] == 2017

    X_train = df.loc[train_mask, features]
    y_train = df.loc[train_mask, "Precio"]
    X_test = df.loc[test_mask, features]
    y_test = df.loc[test_mask, "Precio"]

    print(f"Entrenamiento 2016: {len(X_train):,} filas")
    print(f"Evaluación 2017:    {len(X_test):,} filas")
    print(f"Mediana Precio 2016 (baseline): ${y_train.median():,.0f}")
    print(f"Mediana Precio 2017:            ${y_test.median():,.0f}")

**Paso 6 — Pipeline (sin leakage)**

**¿Qué?** Un `ColumnTransformer`: numéricas con mediana + escalado; categóricas con constante `missing` + one-hot que ignora niveles no vistos.

**¿Por qué?** Mediana, media y categorías se aprenden **solo en 2016**. Si 2017 trae una región nueva, el encoder no inventa una columna con información del test.

In [ ]:
if not DATA_AVAILABLE:
    print("NO VERIFICABLE CON LOS DATOS DISPONIBLES.")
else:
    preprocesador = ColumnTransformer(
        transformers=[
            (
                "num",
                Pipeline([
                    ("imputer", SimpleImputer(strategy="median")),
                    ("scaler", StandardScaler()),
                ]),
                variables_numericas,
            ),
            (
                "cat",
                Pipeline([
                    ("imputer", SimpleImputer(strategy="constant", fill_value="missing")),
                    ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
                ]),
                variables_categoricas,
            ),
        ]
    )
    print("Preprocesador definido. Fit se hará dentro de cada modelo, solo sobre X_train.")

**Paso 7 — Baseline y modelos candidatos**

**¿Qué?** Mediana 2016 → lineal → árbol → Random Forest → Gradient Boosting.

**¿Por qué?** El enunciado pide describir el(los) modelo(s) y el método de ajuste. La complejidad solo se justifica si mejora el test 2017 de forma material, no porque el R² de entrenamiento se vea alto.

In [ ]:
if not DATA_AVAILABLE:
    print("NO VERIFICABLE CON LOS DATOS DISPONIBLES.")
else:
    def con_prep(estimador):
        return Pipeline([("prep", preprocesador), ("reg", estimador)])

    modelos = {
        "Baseline (mediana 2016)": DummyRegressor(strategy="median"),
        "Regresión lineal": con_prep(LinearRegression()),
        "Árbol de decisión": con_prep(
            DecisionTreeRegressor(max_depth=10, min_samples_leaf=15, random_state=RANDOM_STATE)
        ),
        "Random Forest": con_prep(
            RandomForestRegressor(
                n_estimators=150,
                max_depth=16,
                min_samples_leaf=5,
                n_jobs=-1,
                random_state=RANDOM_STATE,
            )
        ),
        "Gradient Boosting": con_prep(
            GradientBoostingRegressor(
                n_estimators=150,
                max_depth=6,
                learning_rate=0.08,
                random_state=RANDOM_STATE,
            )
        ),
    }
    print("Candidatos:", list(modelos))

**Paso 8 — Ajuste en 2016 y evaluación en 2017**

**¿Qué?** Validación cruzada K=5 **dentro de 2016**. Luego un fit con todo 2016 y métricas en 2017.

**¿Por qué?** La CV no sustituye al test temporal. El número que importa para H₁ es el de 2017. Las cifras de esta celda son las del oral: no se copian a mano desde un comentario viejo.

In [ ]:
if not DATA_AVAILABLE:
    print("NO VERIFICABLE CON LOS DATOS DISPONIBLES.")
else:
    cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    filas = []
    predicciones_test = {}

    for nombre, modelo in modelos.items():
        if nombre.startswith("Baseline"):
            cv_r2 = np.nan
            modelo.fit(X_train, y_train)
        else:
            cv_r2 = cross_val_score(modelo, X_train, y_train, cv=cv, scoring="r2", n_jobs=-1).mean()
            modelo.fit(X_train, y_train)

        y_hat_tr = modelo.predict(X_train)
        y_hat_te = modelo.predict(X_test)
        predicciones_test[nombre] = y_hat_te

        filas.append({
            "Modelo": nombre,
            "R² CV 2016": None if np.isnan(cv_r2) else round(float(cv_r2), 4),
            "R² train": round(r2_score(y_train, y_hat_tr), 4),
            "R² test 2017": round(r2_score(y_test, y_hat_te), 4),
            "MAE test (AUD)": round(mean_absolute_error(y_test, y_hat_te), 2),
            "RMSE test (AUD)": round(np.sqrt(mean_squared_error(y_test, y_hat_te)), 2),
            "Δ R² train−test": round(r2_score(y_train, y_hat_tr) - r2_score(y_test, y_hat_te), 4),
        })

    tabla = pd.DataFrame(filas)
    print("================ TABLA COMPARATIVA (esta corrida) ================")
    print(tabla.to_string(index=False))

    mae_base = tabla.loc[tabla["Modelo"].str.startswith("Baseline"), "MAE test (AUD)"].iloc[0]
    mae_rf = tabla.loc[tabla["Modelo"] == "Random Forest", "MAE test (AUD)"].iloc[0]
    print(f"\nReducción MAE RF vs mediana: {(mae_base - mae_rf) / mae_base:.1%}")

**Paso 9 — Selección: Random Forest (no el R² máximo)**

**¿Qué?** El modelo del proyecto es **Random Forest**. Gradient Boosting suele ganar el R² de test; no es el cierre del oral.

**¿Por qué?** El Hito 2 prioriza generalización temporal y estabilidad ante el desplazamiento geográfico, no un concurso de score. La tabla de la celda anterior es la evidencia; los comentarios no pueden citar otra corrida.

In [ ]:
if not DATA_AVAILABLE:
    print("NO VERIFICABLE CON LOS DATOS DISPONIBLES.")
else:
    NOMBRE_FINAL = "Random Forest"
    rf_modelo = modelos[NOMBRE_FINAL]
    y_pred_rf = predicciones_test[NOMBRE_FINAL]
    residuos = y_test.to_numpy() - y_pred_rf

    r2_te = r2_score(y_test, y_pred_rf)
    mae_te = mean_absolute_error(y_test, y_pred_rf)
    rmse_te = np.sqrt(mean_squared_error(y_test, y_pred_rf))
    r2_tr = r2_score(y_train, rf_modelo.predict(X_train))

    print(f"Modelo del Hito 2: {NOMBRE_FINAL}")
    print(f"R² train 2016: {r2_tr:.4f}")
    print(f"R² test 2017:  {r2_te:.4f}")
    print(f"MAE test:      ${mae_te:,.2f}")
    print(f"RMSE test:     ${rmse_te:,.2f}")
    print(
        "Interpretación: H₁ agregada se sostiene si este MAE es claramente menor "
        "que el de la mediana 2016. El error absoluto sigue siendo de negocio: "
        "no es una tasación automática.")
    print("GB, si gana R², se reporta como mejor score, no como modelo elegido.")

**Paso 10 — Diagnóstico de residuos (test 2017)**

**¿Qué?** Observado vs predicho y histograma de residuos.

**¿Por qué?** El R² promedio esconde la cola: el modelo suele fallar más en propiedades de alto valor (>$3M), coherente con el boxplot de `Tipo` del Hito 1.

In [ ]:
if not DATA_AVAILABLE:
    print("NO VERIFICABLE CON LOS DATOS DISPONIBLES.")
else:
    fig, ax = plt.subplots(1, 2, figsize=(14, 5))
    ax[0].scatter(y_test / 1e6, y_pred_rf / 1e6, alpha=0.25, s=12, color="steelblue", edgecolor="none")
    ax[0].plot([0, 9], [0, 9], "--r", lw=2, label="y = x")
    ax[0].set_title("Observado vs predicho — RF — test 2017")
    ax[0].set_xlabel("Precio observado (millones AUD)")
    ax[0].set_ylabel("Precio predicho (millones AUD)")
    ax[0].legend()

    sns.histplot(residuos / 1e3, kde=True, ax=ax[1], color="coral", bins=50)
    ax[1].axvline(0, color="black", ls="--")
    ax[1].set_title("Residuos (real − predicho)")
    ax[1].set_xlabel("Error (miles AUD)")
    plt.tight_layout()
    plt.show()

    cola = (y_test >= 3_000_000).to_numpy()
    yt = y_test.to_numpy()
    print(f"Filas test con Precio ≥ $3M: {int(cola.sum())}")
    if cola.any():
        print(f"MAE en cola ≥ $3M: ${mean_absolute_error(yt[cola], y_pred_rf[cola]):,.0f}")
        print(f"MAE en resto:      ${mean_absolute_error(yt[~cola], y_pred_rf[~cola]):,.0f}")

**Paso 11 — Importancia de variables (interpretación, no causalidad)**

**¿Qué?** Top 10 del Random Forest por reducción de impureza.

**¿Por qué?** Para el oral: qué usa el modelo. No es un efecto causal ni se usa para reelegir variables mirando 2017.

In [ ]:
if not DATA_AVAILABLE:
    print("NO VERIFICABLE CON LOS DATOS DISPONIBLES.")
else:
    rf_reg = rf_modelo.named_steps["reg"]
    enc = rf_modelo.named_steps["prep"].named_transformers_["cat"].named_steps["encoder"]
    nombres = variables_numericas + enc.get_feature_names_out(variables_categoricas).tolist()
    imp = (
        pd.DataFrame({"Variable": nombres, "Importancia": rf_reg.feature_importances_})
        .sort_values("Importancia", ascending=False)
        .head(10)
    )
    print(imp.to_string(index=False))
    plt.figure(figsize=(9, 4.5))
    sns.barplot(data=imp, x="Importancia", y="Variable", color="steelblue")
    plt.title("Top 10 — importancia RF (no causal)")
    plt.tight_layout()
    plt.show()

**Paso 12 — Dónde se rompe: barrios vistos vs no vistos (hipótesis secundaria del Hito 1)**

**¿Qué?** Partir el **mismo** Random Forest en el test 2017: MAE y R² en barrios que ya existían en 2016 versus los 172 nuevos. No se reentrena.

**¿Por qué?** El EDA advirtió el desplazamiento geográfico y no lo midió con error de modelo. Eso es resultado del Hito 2, no un modelo nuevo.

In [ ]:
if not DATA_AVAILABLE:
    print("NO VERIFICABLE CON LOS DATOS DISPONIBLES.")
else:
    test_barrio = df.loc[test_mask, "Barrio"]
    es_nuevo = test_barrio.isin(barrios_nuevos_2017).to_numpy()

    def bloque(nombre, mask):
        n = int(mask.sum())
        if n == 0:
            print(nombre, "sin filas")
            return
        yt, yp = y_test.to_numpy()[mask], y_pred_rf[mask]
        print(
            f"{nombre:28} n={n:5,}  MAE=${mean_absolute_error(yt, yp):>12,.0f}  "
            f"RMSE=${np.sqrt(mean_squared_error(yt, yp)):>12,.0f}  "
            f"R²={r2_score(yt, yp):.4f}"
        )

    print("Mismo RF, sin reentrenar:")
    bloque("Test 2017 completo", np.ones(len(y_test), dtype=bool))
    bloque("Barrios vistos en 2016", ~es_nuevo)
    bloque("Barrios NO vistos", es_nuevo)

    print(
        "\nInterpretación: si el MAE en no vistos es peor, H₁ agregada sigue en pie "
        "y la hipótesis secundaria del EDA queda respaldada. No se imputa Comuna ni se toca el CSV."
    )

**Paso 13 — GridSearchCV sobre RF (método de ajuste; no es el resultado)**

**¿Qué?** Búsqueda de `n_estimators`, `max_depth` y `min_samples_leaf` con CV **solo en 2016**, optimizando MAE.

**¿Por qué?** El enunciado pide metodología de ajuste. Si el margen sobre el RF del Paso 9 es mínimo, **no cambia la decisión**. No se elige el modelo mirando 2017 y después «confirmándolo».

In [ ]:
if not DATA_AVAILABLE:
    print("NO VERIFICABLE CON LOS DATOS DISPONIBLES.")
else:
    param_grid = {
        "reg__n_estimators": [100, 200, 300],
        "reg__max_depth": [10, 15, 20],
        "reg__min_samples_leaf": [5, 10, 20],
    }
    grid = GridSearchCV(
        con_prep(RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=-1)),
        param_grid,
        cv=cv,
        scoring="neg_mean_absolute_error",
        n_jobs=-1,
        verbose=1,
    )
    grid.fit(X_train, y_train)
    y_gs = grid.predict(X_test)
    print("Mejores hiperparámetros (CV 2016):", grid.best_params_)
    print(f"MAE CV 2016: ${-grid.best_score_:,.2f}")
    print(f"R² test 2017 (RF grid):  {r2_score(y_test, y_gs):.4f}")
    print(f"MAE test 2017 (RF grid): ${mean_absolute_error(y_test, y_gs):,.2f}")
    print(f"MAE test 2017 (RF base): ${mae_te:,.2f}")
    print("Si la diferencia es de cientos de dólares, el oral mantiene el RF del Paso 9.")

**Paso 14 — Conclusión del Hito 2**

Tres cierres, una historia (guion oral):

1. **H₁ agregada.** El Random Forest entrenado en 2016 predice 2017 mejor que la mediana de 2016. H₀ se rechaza **a nivel de mercado**. Las cifras son las de la tabla ejecutada (Paso 8–9), no un comentario memorizado.
2. **Diseño del Proyecto 3.** Partición temporal, pipeline sin leakage, jerarquía de modelos, RF elegido por generalización (GB puede tener mejor R²).
3. **Límite de negocio.** El MAE sigue siendo material. 2017 no es un año completo. El mapa se corrió (barrios nuevos; el Paso 12 lo cuantifica). La cola de lujo se subestima. No se inventó `Tamaño_Construcción` ni se tocó el CSV.

Pregunta que vale el 30%: no es «¿pueden subir el R²?». Es «¿2016 generaliza a 2017, con qué error, y dónde no?»